# Quick RAG Revision

## Setting up RAG

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

## Testing it

In [7]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from **https://ollama.com/download** for your operating system:
   - **macOS**: download the `.pkg` and install it
   - **Windows**: download the `.msi` and install it
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that the local server is running, use:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use Ollama from Python, install the client:
   ```bash
   pip install ollama
   ```

   Then you can call it like:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```


In [8]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry for **“Olama”** specifically.

If you mean running the course locally, the FAQ says you can run it locally instead of using Codespaces, but you’ll need to set up the required tools yourself: **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. Also, if you run locally, you should **document your setup and keep the environment reproducible**.

If you meant a different tool name, feel free to clarify.


# Asking without tools

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Usually **yes** — but it depends on the course’s enrollment policy and whether it’s still open.\n\nA few things to check:\n- **Enrollment deadline**: Some courses allow late registration, others don’t.\n- **Prerequisites**: You may need prior knowledge or approval.\n- **Seat availability**: The class may be full.\n- **Access method**: If it’s an online course, you may be able to enroll instantly; if it’s through a school or platform, you might need instructor/admin approval.\n\nIf you want, I can help you figure it out if you tell me:\n1. the **course name/platform**, and  \n2. whether you mean **joining as a student now** or just **accessing the materials**.'

The model answers from its general knowledge, something like "it depends on the course" or "check the course website". It doesn't know about our FAQ, so the answer is vague and not helpful. This is exactly why we need RAG, and why we want to hand the model a tool.

# The agent alternative

Instead of running search ourselves, we give the LLM a `search` tool. It decides when to call it and what to search for.

## Defining the tool

First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## Sending the question with the tool

Now we send the same question as before, but this time we include the tool in the request:

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"can I join the course late discovered course join enrollment late registration"}', call_id='call_7YJ3k9X7CBo8tlXIbP27xDU3', name='search', type='function_call', id='fc_04fb30eaa5414e6d006a3921f88e088197a31ce5362d817377', namespace=None, status='completed')]

Look at the output. Instead of a message with the answer, the response contains a function_call entry. The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first.

Look at the arguments too. The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with. So it rewrote our enrollment question into search keywords like "enroll late join course".

## Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our search function, and serialize the result:

In [13]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result:

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

The `call_id` links the tool result to the specific function call the model requested. If the model makes multiple function calls in one turn, each one gets its own `call_id`.

## Asking the model again

We call the API a second time with the expanded history:

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

This time the model has the original question, its own decision to call search, and the FAQ results. It can now produce a proper course-specific answer.

# The Agentic Loop

An agent has three parts:

Instructions, the role and behavior we want. We pass this as the `developer` message. The better the instructions, the better the agent helps.
Tools, the functions the agent can call to carry out the task. For us that's only `search`.
Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

Everything below is the code that wires these three together inside a loop:

## A developer prompt

In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## A function-call helper

In [17]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

## Processing one response

In [18]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment discovered course can I join"}
function_call: search {"query":"course join late enrollment registration discovered course"}
function_call: search {"query":"new student can I join the course after it started"}


## The full agent loop

In [19]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join.

If you want a certificate, make sure you submit your project while submissions are still open. Also, you don’t need a confirmation email to start; you can begin learning and submitting homework as long as the form is open.

If you’d like, I can also point you to the docs or explain how to get started. Are there other areas you want to explore?


## Wrapping it in a function

In [20]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

Try it with a question that has a typo:

In [21]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local model setup Ollama"}
function_call: search {"query":"run Ollama locally course FAQ install local"}
function_call: search {"query":"Ollama local server model download run command"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model the first time and then open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing available models.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = oll

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model the first time and then open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing available models.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error later, restart the server with:\n\n```bash\nnohup oll

Also try the course enrollment question:

In [22]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered the course can I still join enrollment late join after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. The course materials are available, and you can start anytime.

One important note: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

If you’d like, I can also help you figure out the best way to start the course now. Are there other areas you want to explore?


'Yes — you can still join the course. The course materials are available, and you can start anytime.\n\nOne important note: if you want to receive a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you’d like, I can also help you figure out the best way to start the course now. Are there other areas you want to explore?'

## Encouraging multiple searches

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
iteration #2...
function_call: search {"query":"certificate project submit while accepting submissions live cohort peer review course FAQ self-paced join still join discover course"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you only follow it later in self-paced mode, you can still learn from it, but you won’t be eligible for a certificate.

If you'd like, I can also explain how the homework, project, and certificate process works.


"Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. If you only follow it later in self-paced mode, you can still learn from it, but you won’t be eligible for a certificate.\n\nIf you'd like, I can also explain how the homework, project, and certificate process works."

## Restricting off-topic questions

Right now the agent will answer anything. Ask it about chess and it will still try:

In [24]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit FAQ"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening what is queen gambit"}
iteration #3...
ASSISTANT:
“Queen’s gambit” is a chess opening. It starts with:

1. d4 d5  
2. c4

White offers the c-pawn as a gambit to try to gain better control of the center. Black can accept or decline it. It’s one of the oldest and most respected chess openings.

If you meant the Netflix show **The Queen’s Gambit**, that’s a drama about a young chess prodigy.  

Would you like a quick explanation of the opening ideas, or the common moves after 1.d4 d5 2.c4?


'“Queen’s gambit” is a chess opening. It starts with:\n\n1. d4 d5  \n2. c4\n\nWhite offers the c-pawn as a gambit to try to gain better control of the center. Black can accept or decline it. It’s one of the oldest and most respected chess openings.\n\nIf you meant the Netflix show **The Queen’s Gambit**, that’s a drama about a young chess prodigy.  \n\nWould you like a quick explanation of the opening ideas, or the common moves after 1.d4 d5 2.c4?'

We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ:

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course-related FAQ entry for “queen gambit,” so I can’t answer that from the course materials.

If you meant a course topic, could you rephrase it or provide more context? Are there other areas you want to explore?


'I couldn’t find a course-related FAQ entry for “queen gambit,” so I can’t answer that from the course materials.\n\nIf you meant a course topic, could you rephrase it or provide more context? Are there other areas you want to explore?'

# ToyAIKit

In [26]:
!uv add toyaikit

Resolved 127 packages in 184ms
Installed 7 packages in 506ms
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [27]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

## Registering the tool

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

## Letting ToyAIKit generate the schema

In [29]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [30]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [31]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## The chat interface and runner

In [32]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

## Running one prompt

In [33]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


## Cost and tokens

In [34]:
result.cost

CostInfo(input_cost=Decimal('0.0027645'), output_cost=Decimal('0.001521'), total_cost=Decimal('0.0042855'))

In [35]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local install run locally"}', call_id

## Continuing the conversation

In [36]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


## Interactive chat

For a chat-like workflow, run the built-in input loop:

Type questions and get answers. Type "stop" to exit.

In [37]:
runner.run()

-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How do i run a different model?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"run a different model model change c